In [1]:
# ============================================================
# Full-resolution Trilinear registration
#
# Wavelet: DVF推定の入力だけに使用
# Output : Low-res DVF → Full-res DVF → Moving原画像を直接warp
#
# Curriculum : 100 × Image MSE + 0.01 × DVF MSE
# Fine-tune  : Image MSE only
#
# Curriculum : 80,000 epochs
# Fine-tune  : 30,000 epochs
# 進捗      : 1000 epochごとに保存・可視化、経過時間・ETA表示
# ============================================================

import os
os.environ["VXM_BACKEND"] = "pytorch"

from pathlib import Path
import random
import time
import math

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm

import voxelmorph as vxm
from scipy.ndimage import gaussian_filter

backend:pytorch
Pytorch


In [2]:
# ============================================================
# 1. Settings
# ============================================================

from pathlib import Path

SEED = 42

TRAIN_DATA_PATH = Path(
    r"D:\Saito\Data\TrainData_NoBed.npz"
)

FULL_SIZE = (128, 256, 256)
LOW_SIZE = (64, 128, 128)
COARSE_SIZE = (8, 16, 16)

# ------------------------------------------------------------
# Curriculum: 80,000 epochs
# ------------------------------------------------------------

CURRICULUM_EPOCHS = 80000

BATCH_SIZE = 2
CURRICULUM_LR = 1e-4

# Forward: A -> B
FORWARD_IMAGE_WEIGHT = 100.0
FORWARD_DVF_WEIGHT = 0.01

# Reverse: B -> A
REVERSE_IMAGE_WEIGHT = 100.0

# Inverse consistency
INVERSE_WEIGHT_MAX = 1e-3
INVERSE_WARMUP_EPOCHS = 10000

# ------------------------------------------------------------
# Curriculum shift: ±1 -> ±40
# ------------------------------------------------------------

INITIAL_SHIFT = 1
SHIFT_STEP = 1
SHIFT_STEP_EPOCHS = 2000
MAX_SHIFT = 40

# ------------------------------------------------------------
# Curriculum save / display settings
# ------------------------------------------------------------

CHECKPOINT_EVERY = 2000
PRINT_EVERY = 100
VISUALIZE_EVERY = 1000
PLOT_EVERY = 1000

OBS_MOVING_ID = 0
OBS_DEFORMATION_LEVEL = 20

# 完了済み・使用中の beta=1e-3 curriculum 80k 実験フォルダ
OUTPUT_DIR = Path(
    r"D:\Saito\fullres_trilinear_ic_beta1e3_80k"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROGRESS_DIR = (
    OUTPUT_DIR
    / "curriculum_progress"
)

PROGRESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CURRICULUM_FINAL_PATH = (
    OUTPUT_DIR
    / "model_fullres_trilinear_ic_beta1e3_curriculum_80k_final.pth"
)

print("Curriculum epochs :", CURRICULUM_EPOCHS)
print("Batch size        :", BATCH_SIZE)
print("Output directory  :", OUTPUT_DIR)
print("80k model path    :", CURRICULUM_FINAL_PATH)
print(
    "80k model exists  :",
    CURRICULUM_FINAL_PATH.exists(),
)

Curriculum epochs : 80000
Batch size        : 2
Output directory  : D:\Saito\fullres_trilinear_ic_beta1e3_80k
80k model path    : D:\Saito\fullres_trilinear_ic_beta1e3_80k\model_fullres_trilinear_ic_beta1e3_curriculum_80k_final.pth
80k model exists  : True


In [3]:
# ============================================================
# 2. Device / speed settings
# ============================================================

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

AMP_ENABLED = (device.type == "cuda")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except AttributeError:
    pass

print("Device:", device)
print("AMP enabled:", AMP_ENABLED)

Device: cuda:0
AMP enabled: True


In [4]:

# ============================================================
# 3. Utility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def format_elapsed(seconds):
    seconds = int(seconds)

    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)

    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    return f"{minutes:02d}:{seconds:02d}"


set_seed(SEED)


In [5]:

# ============================================================
# 4. Data
# ============================================================

x_train = np.load(
    TRAIN_DATA_PATH
)["Train"]

# (H, W, D, N) -> (N, D, H, W)
x_train = np.transpose(
    x_train,
    (3, 0, 1, 2),
)

x_train = np.ascontiguousarray(
    x_train,
    dtype=np.float32,
)

if x_train.shape[1:] != FULL_SIZE:
    raise ValueError(
        f"Expected {FULL_SIZE}, got {x_train.shape[1:]}"
    )

if OBS_MOVING_ID >= len(x_train):
    raise IndexError(
        "OBS_MOVING_ID がデータ数を超えています。"
    )

print("x_train:", x_train.shape)


x_train: (400, 128, 256, 256)


In [6]:

# ============================================================
# 5. Fast 3-D Haar analysis
#
# Analysis -> ::2 と同じ出力:
# (B, 1, 128, 256, 256)
# -> (B, 8, 64, 128, 128)
# ============================================================

class Haar3DAnalysisFast(nn.Module):
    def __init__(self):
        super().__init__()

        h_low = torch.tensor(
            [1.0, 1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        h_high = torch.tensor(
            [1.0, -1.0],
            dtype=torch.float32,
        ) / math.sqrt(2.0)

        filters = []

        for z_filter in [h_low, h_high]:
            for y_filter in [h_low, h_high]:
                for x_filter in [h_low, h_high]:
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )

                    filters.append(kernel)

        weight = torch.stack(
            filters,
            dim=0,
        ).unsqueeze(1)

        self.register_buffer(
            "weight",
            weight,
        )

    def forward(self, image):
        image = F.pad(
            image,
            (0, 1, 0, 1, 0, 1),
        )

        return F.conv3d(
            image,
            self.weight,
            stride=2,
            padding=0,
        )


analysis_fast = Haar3DAnalysisFast().to(device)


def wavelet_input(image):
    return analysis_fast(image)

In [7]:
# ============================================================
# 6. Spatial transformers / DVF conversion
# ============================================================

transformer256 = vxm.layers.SpatialTransformer(
    FULL_SIZE
).to(device)

transformer_low_ic = vxm.layers.SpatialTransformer(
    LOW_SIZE
).to(device)


def low_dvf_to_full_trilinear(flow_low):
    """
    low DVFの座標単位はfull DVFの半分。
    したがって補間後に値も2倍する。
    """
    return F.interpolate(
        flow_low.float(),
        size=FULL_SIZE,
        mode="trilinear",
        align_corners=False,
    ) * 2.0

c:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [8]:
# ============================================================
# 7. Artificial DVF teacher for curriculum
# ============================================================

def gaussian_smooth_3d(tensor, sigma=2.0):
    tensor_np = tensor.detach().cpu().numpy()

    smoothed_np = gaussian_filter(
        tensor_np,
        sigma=[0, 0, sigma, sigma, sigma],
    )

    return torch.from_numpy(
        smoothed_np
    ).to(
        device=tensor.device,
        dtype=tensor.dtype,
    )


def generate_artificial_dvf(batch_size, shift):
    coarse_flow = (
        torch.rand(
            (batch_size, 3, *COARSE_SIZE),
            device=device,
            dtype=torch.float32,
        ) * 2.0 - 1.0
    ) * float(shift)

    coarse_flow = gaussian_smooth_3d(
        coarse_flow,
        sigma=2.0,
    )

    # 教師Full-res DVF
    true_flow_full = F.interpolate(
        coarse_flow,
        size=FULL_SIZE,
        mode="trilinear",
        align_corners=False,
    )

    # Full -> Low。空間縮小に伴い座標値も1/2
    true_flow_low = F.interpolate(
        true_flow_full,
        size=LOW_SIZE,
        mode="trilinear",
        align_corners=False,
    ) / 2.0

    return true_flow_full, true_flow_low


def curriculum_shift(epoch):
    level = (
        epoch - 1
    ) // SHIFT_STEP_EPOCHS

    return min(
        INITIAL_SHIFT + level * SHIFT_STEP,
        MAX_SHIFT,
    )

In [9]:
# ============================================================
# 8. Model / AMP
# ============================================================

# 既存の大きいモデル構成を維持
nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16],
]


def create_model():
    return vxm.networks.VxmDense_128_256_256(
        FULL_SIZE,
        nb_features,
        int_steps=0,
    ).to(device)


def create_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=AMP_ENABLED,
        )
    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=AMP_ENABLED,
        )

In [10]:

# ============================================================
# 9. Inverse consistency loss
# native low-resolution DVF上で計算
# ============================================================

z = torch.arange(
    LOW_SIZE[0],
    device=device,
    dtype=torch.float32,
)

y = torch.arange(
    LOW_SIZE[1],
    device=device,
    dtype=torch.float32,
)

x = torch.arange(
    LOW_SIZE[2],
    device=device,
    dtype=torch.float32,
)

zz, yy, xx = torch.meshgrid(
    z, y, x,
    indexing="ij",
)

LOW_BASE_GRID = torch.stack(
    [zz, yy, xx],
    dim=0,
).unsqueeze(0)


def current_inverse_weight(epoch):
    warmup_ratio = min(
        epoch / INVERSE_WARMUP_EPOCHS,
        1.0,
    )

    return INVERSE_WEIGHT_MAX * warmup_ratio


def low_valid_mask(flow):
    """
    範囲外を合成lossから除外する。
    """
    sample_position = (
        LOW_BASE_GRID + flow.float()
    )

    valid_z = (
        (sample_position[:, 0] >= 0)
        & (sample_position[:, 0] <= LOW_SIZE[0] - 1)
    )

    valid_y = (
        (sample_position[:, 1] >= 0)
        & (sample_position[:, 1] <= LOW_SIZE[1] - 1)
    )

    valid_x = (
        (sample_position[:, 2] >= 0)
        & (sample_position[:, 2] <= LOW_SIZE[2] - 1)
    )

    return valid_z & valid_y & valid_x


def masked_vector_mse(vector_field, valid_mask):
    """
    vector_field: (B, 3, D, H, W)
    valid_mask : (B, D, H, W)
    """
    per_voxel_mse = (
        vector_field.float()
        .square()
        .mean(dim=1)
    )

    values = per_voxel_mse[valid_mask]

    if values.numel() == 0:
        raise RuntimeError(
            "IC lossの有効voxelがありません。"
        )

    return values.mean()


def inverse_consistency_loss_low(
    flow_m2f_low,
    flow_f2m_low,
):
    """
    flow_m2f + flow_f2m(flow_m2f) -> 0
    flow_f2m + flow_m2f(flow_f2m) -> 0
    """

    flow_m2f_low = flow_m2f_low.float()
    flow_f2m_low = flow_f2m_low.float()

    # Fixed grid -> Moving grid
    flow_f2m_on_fixed = transformer_low_ic(
        flow_f2m_low,
        flow_m2f_low,
    )

    residual_fixed = (
        flow_m2f_low
        + flow_f2m_on_fixed
    )

    valid_fixed = low_valid_mask(
        flow_m2f_low
    )

    loss_fixed = masked_vector_mse(
        residual_fixed,
        valid_fixed,
    )

    # Moving grid -> Fixed grid
    flow_m2f_on_moving = transformer_low_ic(
        flow_m2f_low,
        flow_f2m_low,
    )

    residual_moving = (
        flow_f2m_low
        + flow_m2f_on_moving
    )

    valid_moving = low_valid_mask(
        flow_f2m_low
    )

    loss_moving = masked_vector_mse(
        residual_moving,
        valid_moving,
    )

    raw_ic_loss = 0.5 * (
        loss_fixed + loss_moving
    )

    valid_fraction = 0.5 * (
        valid_fixed.float().mean()
        + valid_moving.float().mean()
    )

    return raw_ic_loss, valid_fraction

In [11]:
# ============================================================
# 10. Progress helpers
# ============================================================

def save_curriculum_plot(history, output_path):
    if not history:
        return

    epochs = [
        item["epoch"]
        for item in history
    ]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 4),
    )

    axes[0].plot(
        epochs,
        [item["total_loss"] for item in history],
        label="Total weighted loss",
    )

    axes[0].plot(
        epochs,
        [item["image_ab_loss"] for item in history],
        label="Forward image loss",
    )

    axes[0].plot(
        epochs,
        [item["dvf_ab_loss"] for item in history],
        label="Forward DVF loss",
    )

    axes[0].plot(
        epochs,
        [item["image_ba_loss"] for item in history],
        label="Reverse image loss",
    )

    axes[0].plot(
        epochs,
        [item["inverse_loss"] for item in history],
        label="IC weighted loss",
    )

    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Weighted loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(
        epochs,
        [item["raw_inverse_loss"] for item in history],
        label="Raw IC MSE",
    )

    axes[1].plot(
        epochs,
        [item["inverse_weight"] for item in history],
        label="IC weight",
    )

    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Value")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(
        "Inverse-consistency curriculum training"
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=150,
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 固定観察ペア
# Moving画像に「滑らかなランダムDVF」をかけて歪ませる
# ============================================================

obs_moving = torch.from_numpy(
    x_train[
        OBS_MOVING_ID:OBS_MOVING_ID + 1
    ]
).unsqueeze(1).to(
    device=device,
    dtype=torch.float32,
)

# 学習時と同じ generate_artificial_dvf を使う
with torch.no_grad():
    obs_true_flow_full, obs_true_flow_low = (
        generate_artificial_dvf(
            batch_size=1,
            shift=OBS_DEFORMATION_LEVEL,
        )
    )

    obs_fixed = transformer256(
        obs_moving,
        obs_true_flow_full,
    )


@torch.no_grad()
def save_curriculum_image(model, epoch):
    was_training = model.training
    model.eval()

    moving_w = wavelet_input(
        obs_moving
    )

    fixed_w = wavelet_input(
        obs_fixed
    )

    flow_low = model(
        moving_w,
        fixed_w,
    )

    flow_full = low_dvf_to_full_trilinear(
        flow_low
    )

    moved = transformer256(
        obs_moving,
        flow_full,
    )

    slice_index = FULL_SIZE[0] // 2

    moving_np = (
        obs_moving[0, 0, slice_index]
        .float()
        .cpu()
        .numpy()
    )

    fixed_np = (
        obs_fixed[0, 0, slice_index]
        .float()
        .cpu()
        .numpy()
    )

    moved_np = (
        moved[0, 0, slice_index]
        .float()
        .cpu()
        .numpy()
    )

    vmin, vmax = np.percentile(
        np.concatenate(
            [
                moving_np.ravel(),
                fixed_np.ravel(),
                moved_np.ravel(),
            ]
        ),
        [1, 99],
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4),
        constrained_layout=True,
    )

    images = [
        moving_np,
        fixed_np,
        moved_np,
    ]

    titles = [
        "Moving",
        f"Teacher: smooth random DVF (level ±{OBS_DEFORMATION_LEVEL})",
        "Moved prediction",
    ]

    for axis, image, title in zip(
        axes,
        images,
        titles,
    ):
        axis.imshow(
            image,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
        )

        axis.set_title(title)
        axis.axis("off")

    fig.suptitle(
        f"Curriculum epoch {epoch} / "
        f"low DVF {tuple(flow_low.shape)}"
    )

    output_path = (
        PROGRESS_DIR
        / f"curriculum_image_epoch_{epoch:05d}.png"
    )

    fig.savefig(
        output_path,
        dpi=150,
    )

    plt.show()
    plt.close(fig)

    if was_training:
        model.train()

In [12]:
# ============================================================
# 13. 80k curriculum weights -> smoothness fine-tune comparison
#
# Candidates:
#   smoothness weight = 0.001 / 0.01 / 0.1
#
# Loss:
#   symmetric image MSE
#   + inverse-consistency loss
#   + smoothness loss
# ============================================================

import gc

for name in [
    "model3D",
    "optimizer",
    "scaler",
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU cache cleared.")

import csv
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm


GPU cache cleared.


In [13]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

FINETUNE_EPOCHS = 30000
FINETUNE_LR = 1e-6

# 比較する滑らかさlossの重み
SMOOTH_WEIGHTS = [
    0.005,
]

USE_TORCH_COMPILE = False
TORCH_COMPILE_MODE = "reduce-overhead"

# curriculumで使ったIC重みを引き継ぐ
FINETUNE_INVERSE_WEIGHT = INVERSE_WEIGHT_MAX

FINETUNE_CHECKPOINT_EVERY = 2000
FINETUNE_PRINT_EVERY = 100

# 各候補で同じ乱数系列を使う。
# 係数以外の条件を揃えるため。
FINETUNE_SEED = 20260908

CANDIDATE_ROOT = (
    OUTPUT_DIR
    / "finetune_smoothness_comparison"
)

CANDIDATE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if not CURRICULUM_FINAL_PATH.exists():
    raise FileNotFoundError(
        "80k curriculum重みが見つかりません:\n"
        f"{CURRICULUM_FINAL_PATH.resolve()}"
    )

print("=" * 70)
print("SMOOTHNESS FINE-TUNING COMPARISON")
print("=" * 70)
print("80k checkpoint :", CURRICULUM_FINAL_PATH)
print("Epochs         :", FINETUNE_EPOCHS)
print("Learning rate  :", FINETUNE_LR)
print("IC weight      :", FINETUNE_INVERSE_WEIGHT)
print("Smooth weights :", SMOOTH_WEIGHTS)
print("Output root    :", CANDIDATE_ROOT)



SMOOTHNESS FINE-TUNING COMPARISON
80k checkpoint : D:\Saito\fullres_trilinear_ic_beta1e3_80k\model_fullres_trilinear_ic_beta1e3_curriculum_80k_final.pth
Epochs         : 30000
Learning rate  : 1e-06
IC weight      : 0.001
Smooth weights : [0.005]
Output root    : D:\Saito\fullres_trilinear_ic_beta1e3_80k\finetune_smoothness_comparison


In [14]:
# ------------------------------------------------------------
# Smoothness loss
# low-resolution DVFの隣接voxel間の変化を小さくする。
# 小さいほど滑らかで、foldingを抑えやすい。
# ------------------------------------------------------------

def smoothness_loss(flow):
    """
    flow:
        (B, 3, D, H, W)

    Returns:
        DVFの1次微分の二乗平均
    """
    diff_z = (
        flow[:, :, 1:, :, :]
        - flow[:, :, :-1, :, :]
    )

    diff_y = (
        flow[:, :, :, 1:, :]
        - flow[:, :, :, :-1, :]
    )

    diff_x = (
        flow[:, :, :, :, 1:]
        - flow[:, :, :, :, :-1]
    )

    return (
        diff_z.float().square().mean()
        + diff_y.float().square().mean()
        + diff_x.float().square().mean()
    )



In [15]:
# ------------------------------------------------------------
# Checkpoint load
# ------------------------------------------------------------

try:
    curriculum_checkpoint = torch.load(
        CURRICULUM_FINAL_PATH,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    curriculum_checkpoint = torch.load(
        CURRICULUM_FINAL_PATH,
        map_location=device,
    )

curriculum_state_dict = (
    curriculum_checkpoint["model_state_dict"]
)

candidate_results = []


In [16]:
import time
import torch

print(
    "80k model path:",
    CURRICULUM_FINAL_PATH,
    flush=True,
)

print(
    "exists:",
    CURRICULUM_FINAL_PATH.exists(),
    flush=True,
)

if not CURRICULUM_FINAL_PATH.exists():
    raise FileNotFoundError(
        CURRICULUM_FINAL_PATH
    )

file_size_gb = (
    CURRICULUM_FINAL_PATH.stat().st_size
    / 1024**3
)

print(
    f"checkpoint size: {file_size_gb:.3f} GB",
    flush=True,
)

print(
    "torch.load開始...",
    flush=True,
)

start_time = time.perf_counter()

try:
    test_checkpoint = torch.load(
        CURRICULUM_FINAL_PATH,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    test_checkpoint = torch.load(
        CURRICULUM_FINAL_PATH,
        map_location="cpu",
    )

elapsed = time.perf_counter() - start_time

print(
    f"読み込み完了: {elapsed:.1f} 秒",
    flush=True,
)

print(
    "checkpoint keys:",
    list(test_checkpoint.keys()),
    flush=True,
)

del test_checkpoint

80k model path: D:\Saito\fullres_trilinear_ic_beta1e3_80k\model_fullres_trilinear_ic_beta1e3_curriculum_80k_final.pth
exists: True
checkpoint size: 0.093 GB
torch.load開始...
読み込み完了: 0.4 秒
checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'history', 'config', 'architecture']


In [17]:
print("device:", device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

device: cuda:0
CUDA available: True
GPU: NVIDIA GeForce RTX 3090


In [18]:
# ------------------------------------------------------------
# Fine-tune each smoothness candidate
# ------------------------------------------------------------

def state_dict_for_saving(model):
    """
    torch.compileで包まれたモデルでも、
    通常モデルとして再読み込みできるstate_dictを返す。
    """
    if hasattr(model, "_orig_mod"):
        return model._orig_mod.state_dict()

    return model.state_dict()

for smooth_weight in SMOOTH_WEIGHTS:

    # 同一のデータ選択順にして、係数だけを比較する
    set_seed(FINETUNE_SEED)

    candidate_name = (
        f"smooth_{smooth_weight:.0e}"
    )

    candidate_dir = (
        CANDIDATE_ROOT
        / candidate_name
    )

    candidate_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_model_path = (
        candidate_dir
        / (
            "model_fullres_trilinear_ic_"
            f"{candidate_name}_finetuned_"
            f"{FINETUNE_EPOCHS}k_final.pth"
        )
    )

    history_path = (
        candidate_dir
        / "finetune_history.csv"
    )

    print("\n" + "=" * 70)
    print(
        f"Fine-tuning started: "
        f"SMOOTH_WEIGHT = {smooth_weight}"
    )
    print("=" * 70)

    model3D = create_model()

    # compile前に通常モデルへ80k重みを読み込む
    model3D.load_state_dict(
        curriculum_state_dict,
        strict=True,
    )

    if USE_TORCH_COMPILE:
        if not hasattr(torch, "compile"):
            raise RuntimeError(
                "このPyTorchには torch.compile がありません。"
            )

        print(
            f"torch.compile開始: {TORCH_COMPILE_MODE}"
        )

        model3D = torch.compile(
            model3D,
            mode=TORCH_COMPILE_MODE,
        )

    model3D.train()

    optimizer = optim.Adam(
        model3D.parameters(),
        lr=FINETUNE_LR,
    )

    scaler = create_scaler()

    history = []
    start_time = time.perf_counter()

    progress_bar = tqdm(
        range(1, FINETUNE_EPOCHS + 1),
        desc=(
            f"Smooth={smooth_weight:.0e}"
        ),
    )

    for epoch in progress_bar:

        model3D.train()

        moving_ids = np.random.choice(
            len(x_train),
            size=BATCH_SIZE,
            replace=False,
        )

        fixed_ids = np.random.choice(
            len(x_train),
            size=BATCH_SIZE,
            replace=False,
        )

        while np.any(
            moving_ids == fixed_ids
        ):
            same_case = (
                moving_ids == fixed_ids
            )

            fixed_ids[same_case] = (
                np.random.choice(
                    len(x_train),
                    size=same_case.sum(),
                    replace=False,
                )
            )

        moving_images = torch.from_numpy(
            x_train[moving_ids]
        ).unsqueeze(1).to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        fixed_images = torch.from_numpy(
            x_train[fixed_ids]
        ).unsqueeze(1).to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            moving_w = wavelet_input(
                moving_images
            )

            fixed_w = wavelet_input(
                fixed_images
            )

            # Moving -> Fixed
            flow_m2f_low = model3D(
                moving_w,
                fixed_w,
            )

            flow_m2f_full = (
                low_dvf_to_full_trilinear(
                    flow_m2f_low
                )
            )

            moved_ab = transformer256(
                moving_images,
                flow_m2f_full,
            )

            # Fixed -> Moving
            flow_f2m_low = model3D(
                fixed_w,
                moving_w,
            )

            flow_f2m_full = (
                low_dvf_to_full_trilinear(
                    flow_f2m_low
                )
            )

            moved_ba = transformer256(
                fixed_images,
                flow_f2m_full,
            )

        image_loss = 0.5 * (
            F.mse_loss(
                moved_ab.float(),
                fixed_images.float(),
            )
            + F.mse_loss(
                moved_ba.float(),
                moving_images.float(),
            )
        )

        raw_ic_loss, valid_fraction = (
            inverse_consistency_loss_low(
                flow_m2f_low,
                flow_f2m_low,
            )
        )

        raw_smooth_loss = 0.5 * (
            smoothness_loss(flow_m2f_low)
            + smoothness_loss(flow_f2m_low)
        )

        inverse_loss = (
            FINETUNE_INVERSE_WEIGHT
            * raw_ic_loss
        )

        weighted_smooth_loss = (
            smooth_weight
            * raw_smooth_loss
        )

        total_loss = (
            image_loss
            + inverse_loss
            + weighted_smooth_loss
        )

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        history.append(
            {
                "epoch": epoch,
                "total_loss": total_loss.item(),
                "image_loss": image_loss.item(),
                "raw_ic_loss": raw_ic_loss.item(),
                "inverse_loss": inverse_loss.item(),
                "raw_smooth_loss": (
                    raw_smooth_loss.item()
                ),
                "weighted_smooth_loss": (
                    weighted_smooth_loss.item()
                ),
                "ic_valid_fraction": (
                    valid_fraction.item()
                ),
            }
        )

        if (
            epoch == 1
            or epoch % FINETUNE_PRINT_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - start_time
            )

            eta = (
                elapsed / epoch
                * (
                    FINETUNE_EPOCHS
                    - epoch
                )
            )

            progress_bar.set_postfix(
                image=f"{image_loss.item():.6f}",
                ic=f"{raw_ic_loss.item():.6f}",
                smooth=(
                    f"{raw_smooth_loss.item():.6f}"
                ),
                eta=format_elapsed(eta),
            )

            print(
                f"{candidate_name} | "
                f"{epoch:05d}/{FINETUNE_EPOCHS} | "
                f"total={total_loss.item():.7f} | "
                f"image={image_loss.item():.7f} | "
                f"raw_ic={raw_ic_loss.item():.7f} | "
                f"raw_smooth="
                f"{raw_smooth_loss.item():.7f}"
            )

        if (
            epoch % FINETUNE_CHECKPOINT_EVERY
            == 0
        ):
            checkpoint_path = (
                candidate_dir
                / f"epoch_{epoch:05d}.pth"
            )

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": (
                        state_dict_for_saving(model3D)
                    ),
                    "optimizer_state_dict": (
                        optimizer.state_dict()
                    ),
                    "scaler_state_dict": (
                        scaler.state_dict()
                    ),
                    "history": history,
                    "config": {
                        "smooth_weight": (
                            smooth_weight
                        ),
                        "inverse_weight": (
                            FINETUNE_INVERSE_WEIGHT
                        ),
                        "finetune_lr": (
                            FINETUNE_LR
                        ),
                        "finetune_epochs": (
                            FINETUNE_EPOCHS
                        ),
                        "integration_steps": 0,
                    },
                },
                checkpoint_path,
            )

    torch.save(
        {
            "epoch": FINETUNE_EPOCHS,
            "model_state_dict": (
                state_dict_for_saving(model3D)
            ),
            "optimizer_state_dict": (
                optimizer.state_dict()
            ),
            "scaler_state_dict": (
                scaler.state_dict()
            ),
            "history": history,
            "config": {
                "smooth_weight": smooth_weight,
                "inverse_weight": (
                    FINETUNE_INVERSE_WEIGHT
                ),
                "finetune_lr": FINETUNE_LR,
                "finetune_epochs": (
                    FINETUNE_EPOCHS
                ),
                "integration_steps": 0,
            },
            "architecture": (
                "wavelet_dvf_fullres_trilinear_ic"
            ),
            "loss": (
                "symmetric_image_mse_plus_ic_"
                "plus_smoothness"
            ),
        },
        final_model_path,
    )

    with history_path.open(
        "w",
        newline="",
        encoding="utf-8-sig",
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=list(history[0].keys()),
        )

        writer.writeheader()
        writer.writerows(history)

    elapsed = (
        time.perf_counter()
        - start_time
    )

    candidate_results.append(
        {
            "smooth_weight": smooth_weight,
            "final_model_path": str(
                final_model_path.resolve()
            ),
            "training_time": format_elapsed(
                elapsed
            ),
            "final_image_loss": history[-1][
                "image_loss"
            ],
            "final_raw_ic_loss": history[-1][
                "raw_ic_loss"
            ],
            "final_raw_smooth_loss": history[-1][
                "raw_smooth_loss"
            ],
        }
    )

    print(
        f"\nSaved: {final_model_path}"
    )

    del model3D
    del optimizer
    del scaler

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()



Fine-tuning started: SMOOTH_WEIGHT = 0.005
[64, 128, 128]


Smooth=5e-03:   0%|          | 0/30000 [00:00<?, ?it/s]

smooth_5e-03 | 00001/30000 | total=0.0193048 | image=0.0176501 | raw_ic=0.8718250 | raw_smooth=0.1565719
smooth_5e-03 | 00100/30000 | total=0.0077160 | image=0.0065701 | raw_ic=0.5392203 | raw_smooth=0.1213176
smooth_5e-03 | 00200/30000 | total=0.0153287 | image=0.0138547 | raw_ic=0.8331815 | raw_smooth=0.1281598
smooth_5e-03 | 00300/30000 | total=0.0110871 | image=0.0100272 | raw_ic=0.5490589 | raw_smooth=0.1021621
smooth_5e-03 | 00400/30000 | total=0.0298184 | image=0.0276242 | raw_ic=1.4381676 | raw_smooth=0.1511994
smooth_5e-03 | 00500/30000 | total=0.0109088 | image=0.0095959 | raw_ic=0.7365326 | raw_smooth=0.1152686
smooth_5e-03 | 00600/30000 | total=0.0135576 | image=0.0118734 | raw_ic=0.9806728 | raw_smooth=0.1406983
smooth_5e-03 | 00700/30000 | total=0.0083874 | image=0.0070615 | raw_ic=0.6881909 | raw_smooth=0.1275509
smooth_5e-03 | 00800/30000 | total=0.0059865 | image=0.0050693 | raw_ic=0.4538288 | raw_smooth=0.0926842
smooth_5e-03 | 00900/30000 | total=0.0062041 | image=0.

KeyboardInterrupt: 

In [ ]:

# ------------------------------------------------------------
# Candidate list for the TestData evaluation cell
# ------------------------------------------------------------

candidate_summary_path = (
    CANDIDATE_ROOT
    / "candidate_training_summary.csv"
)

with candidate_summary_path.open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=list(
            candidate_results[0].keys()
        ),
    )

    writer.writeheader()
    writer.writerows(candidate_results)

print("\n" + "=" * 70)
print("ALL SMOOTHNESS CANDIDATES COMPLETED")
print("=" * 70)

for row in candidate_results:
    print(
        f"SMOOTH_WEIGHT={row['smooth_weight']:.0e}"
    )
    print(
        "  model:",
        row["final_model_path"],
    )

print(
    "\nTraining summary:",
    candidate_summary_path.resolve(),
)


ALL SMOOTHNESS CANDIDATES COMPLETED
SMOOTH_WEIGHT=2e-03
  model: D:\Saito\fullres_trilinear_ic_beta1e3_80k\finetune_smoothness_comparison\smooth_2e-03\model_fullres_trilinear_ic_smooth_2e-03_finetuned_5000k_final.pth
SMOOTH_WEIGHT=3e-03
  model: D:\Saito\fullres_trilinear_ic_beta1e3_80k\finetune_smoothness_comparison\smooth_3e-03\model_fullres_trilinear_ic_smooth_3e-03_finetuned_5000k_final.pth
SMOOTH_WEIGHT=5e-03
  model: D:\Saito\fullres_trilinear_ic_beta1e3_80k\finetune_smoothness_comparison\smooth_5e-03\model_fullres_trilinear_ic_smooth_5e-03_finetuned_5000k_final.pth
SMOOTH_WEIGHT=7e-03
  model: D:\Saito\fullres_trilinear_ic_beta1e3_80k\finetune_smoothness_comparison\smooth_7e-03\model_fullres_trilinear_ic_smooth_7e-03_finetuned_5000k_final.pth
SMOOTH_WEIGHT=8e-03
  model: D:\Saito\fullres_trilinear_ic_beta1e3_80k\finetune_smoothness_comparison\smooth_8e-03\model_fullres_trilinear_ic_smooth_8e-03_finetuned_5000k_final.pth

Training summary: D:\Saito\fullres_trilinear_ic_beta1e3_8